# MBPO on HalfCheetah-v5 — training notebook (Group 18)

**Model-Based Policy Optimization** (Janner et al., 2019) on Gymnasium's MuJoCo `HalfCheetah-v5`.

Team: Azamat Kharisov, Alexander Kurilenko, Egor Novokreshchenov, Ivan Prikhodko, Maxim Izyumov.

### How to run on Kaggle
1. *Settings → Accelerator →* **GPU T4 x2**, and *Settings → Internet →* **On** (needed for `pip install`).
2. Click **Save Version → Save & Run All (Commit)**. The commit runs in the background for up to 12 h,
   even with the browser closed, and keeps everything written to `/kaggle/working`.
3. When it finishes, open the version's **Output** tab and download `mbpo_results.zip`
   (learning curves, logs, TensorBoard files, checkpoints, videos).

### What this notebook does
| step | what | time |
|---|---|---|
| 1 | install `gymnasium[mujoco]`, check GPUs | ~2 min |
| 2 | write the `mbpo` package to disk (`%%writefile` cells below) | instant |
| 3 | check MuJoCo off-screen rendering, record a *random-policy* video | ~1 min |
| 4 | 1-minute smoke test of the full pipeline (fails fast if anything is broken) | ~2 min |
| 5 | **train**: MBPO seed 0 on GPU 0, MBPO seed 1 on GPU 1, SAC baseline on GPU 0 | `TRAIN_HOURS` |
| 6 | plots, final evaluation table, video of the trained cheetah, zip of all results | ~5 min |

The two GPUs are used to run **two MBPO seeds in parallel** (MBPO is a sequential algorithm, so one run
cannot use two GPUs efficiently); the cheap model-free SAC baseline shares GPU 0 and shows the
sample-efficiency gain.

In [ ]:
# ================= settings =================
import os
TRAIN_HOURS = 11.0          # wall-clock budget per run; Kaggle kills a session at 12 h
TOTAL_ENV_STEPS = 400_000   # paper budget for HalfCheetah; the time budget usually stops us first
UPDATES_PER_STEP = 20       # G: SAC gradient steps per env step (paper: 40; 20 to fit in 12 h)
MBPO_SEEDS = [0, 1]         # one run per GPU
RUN_SAC_BASELINE = True     # model-free SAC with the same env-step budget
SMOKE_TEST = os.environ.get("MBPO_SMOKE_TEST") == "1"  # set True for a ~3-minute dry run only
WORK_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
os.chdir(WORK_DIR)
print("working directory:", WORK_DIR, "| smoke test only:", SMOKE_TEST)

In [ ]:
%pip install -q "gymnasium[mujoco]>=1.0" imageio imageio-ffmpeg

In [ ]:
import subprocess, sys, torch, gymnasium, mujoco
print("python", sys.version.split()[0], "| torch", torch.__version__,
      "| gymnasium", gymnasium.__version__, "| mujoco", mujoco.__version__)
N_GPU = torch.cuda.device_count()
for i in range(N_GPU):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
if N_GPU == 0:
    print("WARNING: no GPU found - enable 'GPU T4 x2' in the notebook settings (training on CPU is far too slow).")

## The implementation
Each cell below writes one source file. The same files live in our repository; the notebook just
bundles them so that Kaggle needs nothing but this `.ipynb`.

In [ ]:
import os
os.makedirs('mbpo', exist_ok=True)

**`mbpo/__init__.py`** — Package entry point.

In [ ]:
%%writefile mbpo/__init__.py
"""Model-Based Policy Optimization (MBPO) for HalfCheetah, in PyTorch."""
from .config import Config, mbpo_config, sac_config, smoke_config
from .dynamics import ProbabilisticEnsemble
from .replay_buffer import ReplayBuffer
from .sac import SAC
from .trainer import MBPOTrainer, evaluate, make_env

__all__ = ["Config", "mbpo_config", "sac_config", "smoke_config", "ProbabilisticEnsemble",
           "ReplayBuffer", "SAC", "MBPOTrainer", "evaluate", "make_env"]

**`mbpo/config.py`** — All hyper-parameters (paper defaults for HalfCheetah).

In [ ]:
%%writefile mbpo/config.py
"""Hyper-parameters for MBPO and the model-free SAC baseline.

Defaults follow the official MBPO HalfCheetah configuration
(Janner et al., 2019, `examples/config/halfcheetah.py`) unless a comment says
otherwise. The main deliberate deviation is `updates_per_step` (G): the paper
uses 40 SAC updates per environment step; we default to 20 so that a run fits
in a single 12-hour Kaggle session.
"""
from __future__ import annotations

import dataclasses
import json
from dataclasses import dataclass, field
from typing import List, Optional


@dataclass
class Config:
    # ---- experiment -------------------------------------------------------
    algo: str = "mbpo"                  # "mbpo" or "sac" (model-free baseline)
    env_id: str = "HalfCheetah-v5"
    seed: int = 0
    device: str = "auto"                # "auto" -> cuda if available, else cpu
    out_dir: str = "runs/mbpo_seed0"

    # ---- budget -----------------------------------------------------------
    total_env_steps: int = 400_000      # paper: 400k for HalfCheetah
    time_budget_hours: Optional[float] = None  # stop cleanly when exceeded
    epoch_length: int = 1_000           # env steps per "epoch" (one eval each)
    init_random_steps: int = 5_000      # uniform-random exploration at start

    # ---- SAC --------------------------------------------------------------
    hidden_dim: int = 256
    gamma: float = 0.99
    tau: float = 5e-3
    actor_lr: float = 3e-4
    critic_lr: float = 3e-4
    alpha_lr: float = 3e-4
    init_alpha: float = 1.0
    target_entropy: Optional[float] = -3.0  # MBPO HalfCheetah; None -> -|A|
    batch_size: int = 256
    updates_per_step: int = 20          # G; paper uses 40 for HalfCheetah
    replay_size: int = 1_000_000

    # ---- dynamics model ---------------------------------------------------
    ensemble_size: int = 7
    num_elites: int = 5
    model_hidden_dim: int = 200
    model_num_layers: int = 4           # hidden layers
    model_lr: float = 1e-3
    model_batch_size: int = 256
    model_weight_decay: List[float] = field(
        default_factory=lambda: [2.5e-5, 5e-5, 7.5e-5, 7.5e-5, 1e-4])
    model_holdout_ratio: float = 0.2
    model_max_holdout: int = 5_000
    model_patience: int = 5             # epochs without >1% holdout improvement
    model_max_epochs: Optional[int] = None  # cap per training call (None = until converged)
    model_train_freq: int = 250         # retrain model every N env steps

    # ---- branched rollouts -----------------------------------------------
    rollout_batch_size: int = 100_000   # number of real start states per rollout phase
    # rollout length schedule [min_epoch, max_epoch, min_len, max_len]:
    # length grows linearly from min_len to max_len between the two epochs.
    rollout_schedule: List[int] = field(default_factory=lambda: [20, 150, 1, 1])
    model_retain_epochs: int = 1        # how many epochs of model data to keep
    real_ratio: float = 0.05            # fraction of real data in each SAC batch

    # ---- evaluation / logging --------------------------------------------
    eval_episodes: int = 1              # deterministic eval episodes per epoch
    final_eval_episodes: int = 10
    log_every_updates: int = 1_000      # how often to pull SAC stats to the CPU
    save_every_epochs: int = 10
    tensorboard: bool = True

    # ---------------------------------------------------------------------
    @property
    def use_model(self) -> bool:
        return self.algo == "mbpo"

    def to_dict(self) -> dict:
        return dataclasses.asdict(self)

    def save(self, path: str) -> None:
        with open(path, "w") as f:
            json.dump(self.to_dict(), f, indent=2)

    @classmethod
    def from_dict(cls, d: dict) -> "Config":
        names = {f.name for f in dataclasses.fields(cls)}
        return cls(**{k: v for k, v in d.items() if k in names})


def mbpo_config(**overrides) -> Config:
    """MBPO on HalfCheetah (paper defaults, G=20)."""
    return dataclasses.replace(Config(algo="mbpo"), **overrides)


def sac_config(**overrides) -> Config:
    """Plain model-free SAC baseline with standard SAC settings."""
    base = Config(
        algo="sac",
        out_dir="runs/sac_seed0",
        updates_per_step=1,      # standard SAC: one gradient step per env step
        target_entropy=None,     # standard heuristic -|A| = -6
        real_ratio=1.0,
    )
    return dataclasses.replace(base, **overrides)


def smoke_config(algo: str = "mbpo", **overrides) -> Config:
    """Tiny settings for a quick end-to-end check (runs in ~1 minute on CPU)."""
    make = mbpo_config if algo == "mbpo" else sac_config
    small = dict(
        total_env_steps=3_000,
        epoch_length=500,
        init_random_steps=1_000,
        updates_per_step=2 if algo == "mbpo" else 1,
        model_train_freq=250,
        rollout_batch_size=2_000,
        model_max_epochs=5,
        rollout_schedule=[1, 4, 1, 3],
        eval_episodes=1,
        final_eval_episodes=2,
        log_every_updates=100,
        save_every_epochs=2,
    )
    small.update(overrides)
    return make(**small)


PRESETS = {"mbpo": mbpo_config, "sac": sac_config}

**`mbpo/utils.py`** — Seeding, device selection, CSV + TensorBoard logging.

In [ ]:
%%writefile mbpo/utils.py
"""Small helpers: seeding, device selection and a CSV + TensorBoard logger."""
from __future__ import annotations

import csv
import os
import random
from typing import Dict

import numpy as np
import torch


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def get_device(name: str = "auto") -> torch.device:
    if name == "auto":
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    return torch.device(name)


class Logger:
    """Writes one row per epoch to `progress.csv` and mirrors it to TensorBoard."""

    def __init__(self, out_dir: str, use_tensorboard: bool = True):
        os.makedirs(out_dir, exist_ok=True)
        self.csv_path = os.path.join(out_dir, "progress.csv")
        self._rows: list = []
        self._fields: list = []
        self._tb = None
        if use_tensorboard:
            try:
                from torch.utils.tensorboard import SummaryWriter
                self._tb = SummaryWriter(os.path.join(out_dir, "tb"))
            except Exception as e:  # tensorboard not installed -> CSV only
                print(f"[logger] TensorBoard disabled ({e})")

    def log(self, row: Dict[str, float], step: int) -> None:
        # columns can appear later (e.g. model stats after the first model fit),
        # so keep all rows and rewrite the (small) file each time
        self._rows.append(dict(row))
        self._fields += [k for k in row if k not in self._fields]
        with open(self.csv_path, "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=self._fields)
            w.writeheader()
            w.writerows(self._rows)
        if self._tb is not None:
            for k, v in row.items():
                if isinstance(v, (int, float)) and np.isfinite(v):
                    self._tb.add_scalar(k, v, step)
            self._tb.flush()

    def close(self) -> None:
        if self._tb is not None:
            self._tb.close()

**`mbpo/networks.py`** — Batched ensemble layers, twin Q-network, squashed-Gaussian actor.

In [ ]:
%%writefile mbpo/networks.py
"""Neural network building blocks shared by SAC and the dynamics model.

`EnsembleLinear` holds the weights of E independent linear layers in a single
tensor, so E networks are evaluated with one batched matmul instead of a Python
loop. We use it for the probabilistic dynamics ensemble (E = 7) and for SAC's
twin Q-functions (E = 2).
"""
from __future__ import annotations

import math

import torch
import torch.nn as nn
import torch.nn.functional as F


class EnsembleLinear(nn.Module):
    def __init__(self, in_dim: int, out_dim: int, ensemble_size: int):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(ensemble_size, in_dim, out_dim))
        self.bias = nn.Parameter(torch.zeros(ensemble_size, 1, out_dim))
        # same init as nn.Linear (Kaiming-uniform), applied per member
        bound = 1.0 / math.sqrt(in_dim)
        nn.init.uniform_(self.weight, -bound, bound)
        nn.init.uniform_(self.bias, -bound, bound)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (E, B, in_dim) -> (E, B, out_dim)
        return torch.baddbmm(self.bias, x, self.weight)


class EnsembleMLP(nn.Module):
    """E independent MLPs evaluated in parallel."""

    def __init__(self, in_dim, out_dim, hidden_dim, num_hidden, ensemble_size, act=nn.ReLU):
        super().__init__()
        dims = [in_dim] + [hidden_dim] * num_hidden
        self.layers = nn.ModuleList(
            [EnsembleLinear(a, b, ensemble_size) for a, b in zip(dims[:-1], dims[1:])])
        self.out = EnsembleLinear(dims[-1], out_dim, ensemble_size)
        self.act = act()
        self.ensemble_size = ensemble_size

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.dim() == 2:  # same input for every member
            x = x.unsqueeze(0).expand(self.ensemble_size, *x.shape)
        for layer in self.layers:
            x = self.act(layer(x))
        return self.out(x)


class TwinQ(nn.Module):
    """Two Q-networks Q1, Q2 (clipped double-Q) evaluated as one ensemble."""

    def __init__(self, obs_dim, act_dim, hidden_dim=256):
        super().__init__()
        self.net = EnsembleMLP(obs_dim + act_dim, 1, hidden_dim, 2, ensemble_size=2)

    def forward(self, obs, act):
        q = self.net(torch.cat([obs, act], dim=-1))  # (2, B, 1)
        return q.squeeze(-1)                          # (2, B)


class SquashedGaussianActor(nn.Module):
    """pi(a|s) = tanh(N(mu(s), sigma(s)^2)), the standard SAC policy."""

    LOG_STD_MIN, LOG_STD_MAX = -20.0, 2.0

    def __init__(self, obs_dim, act_dim, hidden_dim=256):
        super().__init__()
        self.trunk = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU())
        self.mu = nn.Linear(hidden_dim, act_dim)
        self.log_std = nn.Linear(hidden_dim, act_dim)

    def forward(self, obs, deterministic=False, with_log_prob=True):
        h = self.trunk(obs)
        mu = self.mu(h)
        log_std = self.log_std(h).clamp(self.LOG_STD_MIN, self.LOG_STD_MAX)
        std = log_std.exp()
        u = mu if deterministic else mu + std * torch.randn_like(mu)  # reparameterisation
        a = torch.tanh(u)
        if not with_log_prob:
            return a, None
        # log N(u; mu, std) - sum log(1 - tanh(u)^2), written in a stable form
        log_prob = (-0.5 * ((u - mu) / std) ** 2 - log_std - 0.5 * math.log(2 * math.pi)).sum(-1)
        log_prob = log_prob - (2.0 * (math.log(2.0) - u - F.softplus(-2.0 * u))).sum(-1)
        return a, log_prob

**`mbpo/replay_buffer.py`** — GPU-resident replay buffer (used for both D_env and D_model).

In [ ]:
%%writefile mbpo/replay_buffer.py
"""FIFO replay buffer that lives on the training device.

Keeping the data on the GPU means sampling a mini-batch is a single indexing
op, with no host-to-device copy inside the (very hot) SAC update loop.
"""
from __future__ import annotations

from typing import Dict

import numpy as np
import torch

FIELDS = ("obs", "act", "rew", "next_obs", "done")


class ReplayBuffer:
    def __init__(self, obs_dim: int, act_dim: int, capacity: int, device: torch.device):
        self.capacity = int(capacity)
        self.device = device
        self.obs_dim, self.act_dim = obs_dim, act_dim
        self._alloc(self.capacity)
        self.ptr = 0
        self.size = 0

    def _alloc(self, capacity):
        z = lambda *s: torch.zeros(capacity, *s, dtype=torch.float32, device=self.device)
        self.data = {"obs": z(self.obs_dim), "act": z(self.act_dim), "rew": z(),
                     "next_obs": z(self.obs_dim), "done": z()}

    def __len__(self):
        return self.size

    # -- writing -------------------------------------------------------------
    def add(self, obs, act, rew, next_obs, done) -> None:
        """Add one transition (numpy / python scalars)."""
        self.add_batch({
            "obs": torch.as_tensor(np.asarray(obs)[None], dtype=torch.float32),
            "act": torch.as_tensor(np.asarray(act)[None], dtype=torch.float32),
            "rew": torch.tensor([float(rew)]),
            "next_obs": torch.as_tensor(np.asarray(next_obs)[None], dtype=torch.float32),
            "done": torch.tensor([float(done)]),
        })

    def add_batch(self, batch: Dict[str, torch.Tensor]) -> None:
        n = batch["obs"].shape[0]
        if n >= self.capacity:  # keep only the newest `capacity` items
            batch = {k: v[-self.capacity:] for k, v in batch.items()}
            n = self.capacity
        idx = (torch.arange(n, device=self.device) + self.ptr) % self.capacity
        for k in FIELDS:
            self.data[k][idx] = batch[k].to(self.device, torch.float32).reshape(n, *self.data[k].shape[1:])
        self.ptr = (self.ptr + n) % self.capacity
        self.size = min(self.size + n, self.capacity)

    # -- reading -------------------------------------------------------------
    def sample(self, batch_size: int) -> Dict[str, torch.Tensor]:
        idx = torch.randint(0, self.size, (batch_size,), device=self.device)
        return {k: v[idx] for k, v in self.data.items()}

    def all(self) -> Dict[str, torch.Tensor]:
        """All stored transitions in chronological order."""
        if self.size < self.capacity:
            return {k: v[: self.size] for k, v in self.data.items()}
        order = (torch.arange(self.capacity, device=self.device) + self.ptr) % self.capacity
        return {k: v[order] for k, v in self.data.items()}

    def resize(self, new_capacity: int) -> None:
        """Change capacity, keeping the most recent transitions (used when the
        rollout length, and therefore the amount of model data, changes)."""
        new_capacity = int(new_capacity)
        if new_capacity == self.capacity:
            return
        old = self.all()
        self.capacity = new_capacity
        self._alloc(new_capacity)
        self.ptr = self.size = 0
        if old["obs"].shape[0] > 0:
            self.add_batch(old)

**`mbpo/sac.py`** — Soft Actor-Critic: the policy optimiser inside MBPO.

In [ ]:
%%writefile mbpo/sac.py
"""Soft Actor-Critic (Haarnoja et al., 2018), the policy optimiser inside MBPO.

One call to `update(batch)` performs one gradient step on each of

  critic:  L_Q  = E[(Q_i(s,a) - y)^2],   y = r + gamma (1-d) [min_j Q'_j(s',a') - alpha log pi(a'|s')],  a' ~ pi(.|s')
  actor:   L_pi = E[alpha log pi(a~|s) - min_j Q_j(s,a~)],                                                a~ ~ pi(.|s)
  alpha:   L_a  = E[-log_alpha (log pi(a~|s) + H_target)]

followed by a Polyak update of the target critic: Q' <- tau Q + (1-tau) Q'.
"""
from __future__ import annotations

import copy
from typing import Dict, Optional

import numpy as np
import torch
import torch.nn.functional as F

from .networks import SquashedGaussianActor, TwinQ


def _adam(params, lr, device):
    # the fused CUDA kernel is noticeably faster for these small networks
    kwargs = {"fused": True} if device.type == "cuda" else {}
    try:
        return torch.optim.Adam(params, lr=lr, **kwargs)
    except (TypeError, RuntimeError):
        return torch.optim.Adam(params, lr=lr)


class SAC:
    def __init__(self, obs_dim, act_dim, device, hidden_dim=256, gamma=0.99, tau=5e-3,
                 actor_lr=3e-4, critic_lr=3e-4, alpha_lr=3e-4, init_alpha=1.0,
                 target_entropy: Optional[float] = None):
        self.device = device
        self.gamma, self.tau = gamma, tau
        self.actor = SquashedGaussianActor(obs_dim, act_dim, hidden_dim).to(device)
        self.critic = TwinQ(obs_dim, act_dim, hidden_dim).to(device)
        self.critic_target = copy.deepcopy(self.critic).requires_grad_(False)
        self.log_alpha = torch.tensor(np.log(init_alpha), dtype=torch.float32,
                                      device=device, requires_grad=True)
        self.target_entropy = float(-act_dim if target_entropy is None else target_entropy)

        self.actor_opt = _adam(self.actor.parameters(), actor_lr, device)
        self.critic_opt = _adam(self.critic.parameters(), critic_lr, device)
        self.alpha_opt = _adam([self.log_alpha], alpha_lr, device)
        self._critic_params = list(self.critic.parameters())
        self._target_params = list(self.critic_target.parameters())

    @property
    def alpha(self) -> torch.Tensor:
        return self.log_alpha.exp()

    # ---------------------------------------------------------------------
    @torch.no_grad()
    def act(self, obs: np.ndarray, deterministic: bool = False) -> np.ndarray:
        o = torch.as_tensor(obs, dtype=torch.float32, device=self.device)
        a, _ = self.actor(o.unsqueeze(0) if o.dim() == 1 else o,
                          deterministic=deterministic, with_log_prob=False)
        a = a.cpu().numpy()
        return a[0] if o.dim() == 1 else a

    def update(self, batch: Dict[str, torch.Tensor], return_info: bool = False):
        obs, act, rew = batch["obs"], batch["act"], batch["rew"]
        next_obs, done = batch["next_obs"], batch["done"]
        alpha = self.alpha.detach()

        # ---- critic ------------------------------------------------------
        with torch.no_grad():
            next_a, next_logp = self.actor(next_obs)
            q_next = self.critic_target(next_obs, next_a).min(0).values
            target = rew + self.gamma * (1.0 - done) * (q_next - alpha * next_logp)
        q = self.critic(obs, act)                                 # (2, B)
        critic_loss = F.mse_loss(q, target.unsqueeze(0).expand_as(q), reduction="none").mean(1).sum()
        self.critic_opt.zero_grad(set_to_none=True)
        critic_loss.backward()
        self.critic_opt.step()

        # ---- actor -------------------------------------------------------
        for p in self._critic_params:
            p.requires_grad_(False)
        new_a, logp = self.actor(obs)
        q_pi = self.critic(obs, new_a).min(0).values
        actor_loss = (alpha * logp - q_pi).mean()
        self.actor_opt.zero_grad(set_to_none=True)
        actor_loss.backward()
        self.actor_opt.step()
        for p in self._critic_params:
            p.requires_grad_(True)

        # ---- temperature -------------------------------------------------
        alpha_loss = -(self.log_alpha * (logp.detach() + self.target_entropy)).mean()
        self.alpha_opt.zero_grad(set_to_none=True)
        alpha_loss.backward()
        self.alpha_opt.step()

        # ---- target network ----------------------------------------------
        with torch.no_grad():
            torch._foreach_mul_(self._target_params, 1.0 - self.tau)
            torch._foreach_add_(self._target_params, self._critic_params, alpha=self.tau)

        if return_info:  # tensors; caller converts to floats rarely (avoids GPU syncs)
            return {"critic_loss": critic_loss.detach(), "actor_loss": actor_loss.detach(),
                    "alpha_loss": alpha_loss.detach(), "alpha": alpha,
                    "entropy": -logp.detach().mean(), "q_mean": q.detach().mean()}
        return None

    # ---------------------------------------------------------------------
    def state_dict(self):
        return {"actor": self.actor.state_dict(), "critic": self.critic.state_dict(),
                "critic_target": self.critic_target.state_dict(),
                "log_alpha": self.log_alpha.detach().cpu()}

    def load_state_dict(self, sd):
        self.actor.load_state_dict(sd["actor"])
        self.critic.load_state_dict(sd["critic"])
        self.critic_target.load_state_dict(sd["critic_target"])
        with torch.no_grad():
            self.log_alpha.copy_(sd["log_alpha"].to(self.device))

**`mbpo/dynamics.py`** — Probabilistic ensemble dynamics model p(s', r | s, a).

In [ ]:
%%writefile mbpo/dynamics.py
"""Probabilistic ensemble dynamics model p_theta(s', r | s, a) used by MBPO.

Each of the E members is an MLP that outputs a diagonal Gaussian over
y = [r, s' - s] (reward and state *difference*). Members are trained by
maximum likelihood on bootstrapped copies of the real data, with a held-out
set for early stopping and for choosing the `num_elites` best members.
This mirrors the PETS/MBPO model (Chua et al., 2018; Janner et al., 2019).

Loss for one member (Gaussian negative log-likelihood, constants dropped):

    L = mean_{batch, dims} [ (mu - y)^2 * exp(-logvar) + logvar ]
        + 0.01 * (sum(max_logvar) - sum(min_logvar))      # keep variance bounds tight
        + sum_l  wd_l * ||W_l||^2 / 2                     # per-layer weight decay
"""
from __future__ import annotations

import time
from typing import Dict, List, Optional, Sequence

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from .networks import EnsembleLinear


class StandardScaler:
    """Normalises model inputs to zero mean / unit variance (fit on real data)."""

    def __init__(self, dim: int, device):
        self.mu = torch.zeros(1, dim, device=device)
        self.std = torch.ones(1, dim, device=device)

    def fit(self, x: torch.Tensor) -> None:
        self.mu = x.mean(0, keepdim=True)
        std = x.std(0, keepdim=True)
        self.std = torch.where(std < 1e-12, torch.ones_like(std), std)

    def transform(self, x: torch.Tensor) -> torch.Tensor:
        return (x - self.mu) / self.std


class ProbabilisticEnsemble(nn.Module):
    def __init__(self, obs_dim: int, act_dim: int, ensemble_size: int = 7, num_elites: int = 5,
                 hidden_dim: int = 200, num_layers: int = 4,
                 weight_decay: Sequence[float] = (2.5e-5, 5e-5, 7.5e-5, 7.5e-5, 1e-4),
                 lr: float = 1e-3, device="cpu"):
        super().__init__()
        self.obs_dim, self.act_dim = obs_dim, act_dim
        self.in_dim, self.out_dim = obs_dim + act_dim, obs_dim + 1  # out = [reward, delta_obs]
        self.ensemble_size, self.num_elites = ensemble_size, num_elites
        self.device = torch.device(device)

        dims = [self.in_dim] + [hidden_dim] * num_layers + [2 * self.out_dim]
        self.layers = nn.ModuleList(
            [EnsembleLinear(a, b, ensemble_size) for a, b in zip(dims[:-1], dims[1:])])
        assert len(weight_decay) == len(self.layers), "one weight-decay value per layer"
        self.weight_decay = list(weight_decay)
        # learned soft bounds on the predicted log-variance
        self.max_logvar = nn.Parameter(torch.full((1, 1, self.out_dim), 0.5))
        self.min_logvar = nn.Parameter(torch.full((1, 1, self.out_dim), -10.0))

        self.scaler = StandardScaler(self.in_dim, self.device)
        self.elites: List[int] = list(range(num_elites))
        self.to(self.device)
        self.optimizer = torch.optim.Adam(self.parameters(), lr=lr)

    # ---------------------------------------------------------------------
    def forward(self, x_norm: torch.Tensor):
        """x_norm: (E, B, in_dim) normalised inputs -> mean, logvar: (E, B, out_dim)."""
        h = x_norm
        for layer in self.layers[:-1]:
            h = F.silu(layer(h))  # "swish", as in PETS / MBPO
        out = self.layers[-1](h)
        mean, logvar = out[..., : self.out_dim], out[..., self.out_dim:]
        logvar = self.max_logvar - F.softplus(self.max_logvar - logvar)
        logvar = self.min_logvar + F.softplus(logvar - self.min_logvar)
        return mean, logvar

    def _predict_all(self, obs, act):
        x = self.scaler.transform(torch.cat([obs, act], -1))
        return self(x.unsqueeze(0).expand(self.ensemble_size, *x.shape))

    def _weight_decay_loss(self):
        return sum(wd * 0.5 * (layer.weight ** 2).sum()
                   for wd, layer in zip(self.weight_decay, self.layers))

    # ---------------------------------------------------------------------
    @staticmethod
    def make_targets(obs, rew, next_obs):
        return torch.cat([rew.unsqueeze(-1), next_obs - obs], -1)

    def train_model(self, data: Dict[str, torch.Tensor], batch_size: int = 256,
                    holdout_ratio: float = 0.2, max_holdout: int = 5000,
                    patience: int = 5, max_epochs: Optional[int] = None,
                    max_seconds: Optional[float] = None) -> Dict[str, float]:
        """Fit the ensemble on all real data; returns training statistics."""
        t0 = time.time()
        inputs = torch.cat([data["obs"], data["act"]], -1)
        targets = self.make_targets(data["obs"], data["rew"], data["next_obs"])
        n = inputs.shape[0]

        perm = torch.randperm(n, device=self.device)
        n_hold = min(int(n * holdout_ratio), max_holdout)
        hold_idx, train_idx = perm[:n_hold], perm[n_hold:]
        self.scaler.fit(inputs[train_idx])
        x_train, y_train = self.scaler.transform(inputs[train_idx]), targets[train_idx]
        x_hold = self.scaler.transform(inputs[hold_idx]).unsqueeze(0).expand(self.ensemble_size, -1, -1)
        y_hold = targets[hold_idx]
        n_train = x_train.shape[0]

        # bootstrap: every member trains on its own resample (with replacement)
        boot = torch.randint(0, n_train, (self.ensemble_size, n_train), device=self.device)

        best = self._holdout_mse(x_hold, y_hold)
        best_state = {k: v.detach().clone() for k, v in self.state_dict().items()}
        epochs_since_update, epoch, train_loss = 0, 0, float("nan")
        while True:
            shuffled = torch.gather(boot, 1, torch.argsort(torch.rand_like(boot, dtype=torch.float32), 1))
            losses = []
            for start in range(0, n_train, batch_size):
                idx = shuffled[:, start:start + batch_size]          # (E, b)
                mean, logvar = self(x_train[idx])                     # (E, b, out)
                inv_var = torch.exp(-logvar)
                nll = (((mean - y_train[idx]) ** 2) * inv_var + logvar).mean((1, 2)).sum()
                loss = nll + 0.01 * (self.max_logvar.sum() - self.min_logvar.sum()) \
                    + self._weight_decay_loss()
                self.optimizer.zero_grad(set_to_none=True)
                loss.backward()
                self.optimizer.step()
                losses.append(nll.detach())
            epoch += 1
            train_loss = (torch.stack(losses).mean() / self.ensemble_size).item()

            hold = self._holdout_mse(x_hold, y_hold)
            improved = (best - hold) / best.clamp_min(1e-12) > 0.01
            if improved.any():
                best = torch.where(improved, hold, best)
                state = self.state_dict()
                for k, v in state.items():          # keep the best weights of each member
                    if k.startswith("layers.") and v.shape[0] == self.ensemble_size:
                        best_state[k][improved] = v.detach()[improved]
                epochs_since_update = 0
            else:
                epochs_since_update += 1

            if epochs_since_update > patience:
                break
            if max_epochs is not None and epoch >= max_epochs:
                break
            if max_seconds is not None and time.time() - t0 > max_seconds:
                break

        # restore each member's best weights (variance bounds are shared, keep current)
        current = self.state_dict()
        for k in current:
            if k.startswith("layers."):
                current[k].copy_(best_state[k])
        final = self._holdout_mse(x_hold, y_hold)
        self.elites = torch.argsort(final)[: self.num_elites].tolist()
        return {"model_train_loss": train_loss,
                "model_holdout_mse": final[self.elites].mean().item(),
                "model_epochs": epoch,
                "model_train_time": time.time() - t0,
                "model_data_size": n}

    @torch.no_grad()
    def _holdout_mse(self, x_hold, y_hold) -> torch.Tensor:
        mean, _ = self(x_hold)
        return ((mean - y_hold.unsqueeze(0)) ** 2).mean((1, 2))  # (E,)

    # ---------------------------------------------------------------------
    @torch.no_grad()
    def step(self, obs: torch.Tensor, act: torch.Tensor, deterministic: bool = False):
        """Sample (next_obs, reward) for a batch, each row from a random elite member."""
        mean, logvar = self._predict_all(obs, act)                        # (E, B, out)
        sample = mean if deterministic else mean + torch.randn_like(mean) * (0.5 * logvar).exp()
        elites = torch.as_tensor(self.elites, device=obs.device)
        member = elites[torch.randint(len(self.elites), (obs.shape[0],), device=obs.device)]
        chosen = sample[member, torch.arange(obs.shape[0], device=obs.device)]  # (B, out)
        reward, next_obs = chosen[:, 0], obs + chosen[:, 1:]
        return next_obs, reward

    @torch.no_grad()
    def evaluate_mse(self, data: Dict[str, torch.Tensor]) -> float:
        """MSE of the elite-mean prediction on a batch of real transitions."""
        mean, _ = self._predict_all(data["obs"], data["act"])
        pred = mean[self.elites].mean(0)
        y = self.make_targets(data["obs"], data["rew"], data["next_obs"])
        return ((pred - y) ** 2).mean().item()


def termination_fn(env_id: str, obs: torch.Tensor, act: torch.Tensor, next_obs: torch.Tensor):
    """Known termination rule used inside model rollouts (MBPO's static_fns).
    HalfCheetah never terminates (episodes only end by the 1000-step time limit)."""
    if "HalfCheetah" in env_id or "Swimmer" in env_id:
        return torch.zeros(obs.shape[0], device=obs.device)
    raise NotImplementedError(f"no termination function for {env_id}")

**`mbpo/trainer.py`** — The MBPO loop: fit model -> act -> branched rollouts -> SAC updates.

In [ ]:
%%writefile mbpo/trainer.py
"""The MBPO training loop (Janner et al., 2019, Algorithm 2).

    for each epoch:
        every `model_train_freq` env steps:
            train the dynamics ensemble on D_env                    (step 1)
            branch k-step rollouts from real states, add to D_model (step 3)
        act in the real environment with pi, add to D_env           (step 2)
        G times: SAC update on a batch of 5% D_env + 95% D_model    (step 4)
        evaluate the deterministic policy

With `algo="sac"` the model parts are skipped and every batch comes from D_env,
which gives the model-free SAC baseline from the same code.
"""
from __future__ import annotations

import json
import os
import time
from typing import Dict, List

import gymnasium as gym
import numpy as np
import torch

from .config import Config
from .dynamics import ProbabilisticEnsemble, termination_fn
from .replay_buffer import ReplayBuffer
from .sac import SAC
from .utils import Logger, get_device, set_seed


def make_env(env_id: str, seed: int, render_mode=None):
    env = gym.make(env_id, render_mode=render_mode)
    env.reset(seed=seed)
    env.action_space.seed(seed)
    return env


def rollout_length(cfg: Config, epoch: int) -> int:
    """Linear schedule [min_epoch, max_epoch, min_len, max_len] from the paper."""
    min_epoch, max_epoch, min_len, max_len = cfg.rollout_schedule
    if epoch <= min_epoch:
        return int(min_len)
    frac = min((epoch - min_epoch) / max(max_epoch - min_epoch, 1), 1.0)
    return int(round(min_len + frac * (max_len - min_len)))


def evaluate(agent: SAC, env_id: str, episodes: int, seed: int, env=None) -> Dict[str, float]:
    own_env = env is None
    env = make_env(env_id, seed) if own_env else env
    returns, lengths = [], []
    for ep in range(episodes):
        obs, _ = env.reset(seed=seed + ep)
        done, ret, length = False, 0.0, 0
        while not done:
            obs, r, term, trunc, _ = env.step(agent.act(obs, deterministic=True))
            ret += r
            length += 1
            done = term or trunc
        returns.append(ret)
        lengths.append(length)
    if own_env:
        env.close()
    return {"return_mean": float(np.mean(returns)), "return_std": float(np.std(returns)),
            "returns": returns, "length_mean": float(np.mean(lengths))}


class MBPOTrainer:
    def __init__(self, cfg: Config):
        self.cfg = cfg
        os.makedirs(cfg.out_dir, exist_ok=True)
        cfg.save(os.path.join(cfg.out_dir, "config.json"))
        set_seed(cfg.seed)
        self.device = get_device(cfg.device)

        self.env = make_env(cfg.env_id, cfg.seed)
        self.eval_env = make_env(cfg.env_id, cfg.seed + 10_000)
        self.obs_dim = self.env.observation_space.shape[0]
        self.act_dim = self.env.action_space.shape[0]
        assert np.allclose(self.env.action_space.high, 1.0), "actor outputs actions in [-1, 1]"

        self.agent = SAC(self.obs_dim, self.act_dim, self.device, cfg.hidden_dim, cfg.gamma,
                         cfg.tau, cfg.actor_lr, cfg.critic_lr, cfg.alpha_lr, cfg.init_alpha,
                         cfg.target_entropy)
        self.env_buffer = ReplayBuffer(self.obs_dim, self.act_dim,
                                       min(cfg.replay_size, cfg.total_env_steps), self.device)
        self.model = self.model_buffer = None
        if cfg.use_model:
            self.model = ProbabilisticEnsemble(
                self.obs_dim, self.act_dim, cfg.ensemble_size, cfg.num_elites,
                cfg.model_hidden_dim, cfg.model_num_layers, cfg.model_weight_decay,
                cfg.model_lr, self.device)
            self.model_buffer = ReplayBuffer(self.obs_dim, self.act_dim,
                                             self._model_buffer_capacity(1), self.device)
        self.logger = Logger(cfg.out_dir, cfg.tensorboard)
        self.best_eval = -np.inf

    # ------------------------------------------------------------------
    def _model_buffer_capacity(self, k: int) -> int:
        rollouts_per_epoch = self.cfg.rollout_batch_size * self.cfg.epoch_length / self.cfg.model_train_freq
        return int(k * rollouts_per_epoch * self.cfg.model_retain_epochs)

    @torch.no_grad()
    def rollout_model(self, k: int) -> Dict[str, float]:
        """Branch k-step rollouts of the current policy inside the learned model,
        starting from `rollout_batch_size` states sampled from the real buffer."""
        obs = self.env_buffer.sample(self.cfg.rollout_batch_size)["obs"]
        rewards: List[torch.Tensor] = []
        for _ in range(k):
            act, _ = self.agent.actor(obs, with_log_prob=False)
            next_obs, rew = self.model.step(obs, act)
            done = termination_fn(self.cfg.env_id, obs, act, next_obs)
            self.model_buffer.add_batch({"obs": obs, "act": act, "rew": rew,
                                         "next_obs": next_obs, "done": done})
            rewards.append(rew)
            alive = done < 0.5
            if not alive.any():
                break
            obs = next_obs[alive]
        r = torch.cat(rewards)
        return {"rollout_reward_mean": r.mean().item(), "model_buffer_size": len(self.model_buffer)}

    def sample_batch(self) -> Dict[str, torch.Tensor]:
        bs = self.cfg.batch_size
        if not self.cfg.use_model or len(self.model_buffer) == 0:
            return self.env_buffer.sample(bs)
        n_real = int(bs * self.cfg.real_ratio)
        real, fake = self.env_buffer.sample(n_real), self.model_buffer.sample(bs - n_real)
        return {k: torch.cat([real[k], fake[k]], 0) for k in real}

    def save(self, name: str) -> str:
        path = os.path.join(self.cfg.out_dir, name)
        ckpt = {"agent": self.agent.state_dict(), "config": self.cfg.to_dict(),
                "obs_dim": self.obs_dim, "act_dim": self.act_dim}
        if self.model is not None:
            ckpt["model"] = self.model.state_dict()
            ckpt["model_scaler"] = (self.model.scaler.mu.cpu(), self.model.scaler.std.cpu())
            ckpt["model_elites"] = self.model.elites
        torch.save(ckpt, path)
        return path

    # ------------------------------------------------------------------
    def train(self) -> Dict[str, float]:
        cfg = self.cfg
        t_start = time.time()
        budget = None if cfg.time_budget_hours is None else cfg.time_budget_hours * 3600
        obs, _ = self.env.reset(seed=cfg.seed)
        ep_ret, ep_len, last_train_return = 0.0, 0, float("nan")
        n_updates, model_stats, rollout_stats, sac_info = 0, {}, {}, {}
        k = rollout_length(cfg, 0)
        n_epochs = int(np.ceil(cfg.total_env_steps / cfg.epoch_length))
        total_steps = 0
        stop_reason = "finished"

        print(f"[{cfg.algo}] {cfg.env_id} seed={cfg.seed} device={self.device} "
              f"obs={self.obs_dim} act={self.act_dim} epochs={n_epochs}", flush=True)

        for epoch in range(n_epochs):
            t_epoch = time.time()
            if cfg.use_model:
                new_k = rollout_length(cfg, epoch)
                if new_k != k:
                    k = new_k
                    self.model_buffer.resize(self._model_buffer_capacity(k))

            for _ in range(cfg.epoch_length):
                # (1)+(3) train the model and refresh the model buffer
                if (cfg.use_model and total_steps >= cfg.init_random_steps
                        and total_steps % cfg.model_train_freq == 0):
                    model_stats = self.model.train_model(
                        self.env_buffer.all(), cfg.model_batch_size, cfg.model_holdout_ratio,
                        cfg.model_max_holdout, cfg.model_patience, cfg.model_max_epochs)
                    rollout_stats = self.rollout_model(k)

                # (2) interact with the real environment
                if total_steps < cfg.init_random_steps:
                    action = self.env.action_space.sample()
                else:
                    action = self.agent.act(obs)
                next_obs, reward, terminated, truncated, _ = self.env.step(action)
                # time-limit truncation is NOT a terminal state: we keep bootstrapping
                self.env_buffer.add(obs, action, reward, next_obs, float(terminated))
                obs = next_obs
                ep_ret += reward
                ep_len += 1
                total_steps += 1
                if terminated or truncated:
                    last_train_return = ep_ret
                    obs, _ = self.env.reset()
                    ep_ret, ep_len = 0.0, 0

                # (4) policy optimisation
                if total_steps >= cfg.init_random_steps:
                    for _ in range(cfg.updates_per_step):
                        n_updates += 1
                        want_info = n_updates % cfg.log_every_updates == 0
                        info = self.agent.update(self.sample_batch(), return_info=want_info)
                        if info is not None:
                            sac_info = {k_: v.item() for k_, v in info.items()}

                if total_steps >= cfg.total_env_steps:
                    break

            # ---- end of epoch: evaluate, log, checkpoint --------------------
            ev = evaluate(self.agent, cfg.env_id, cfg.eval_episodes,
                          seed=10_000 + cfg.seed + epoch, env=self.eval_env)
            elapsed = time.time() - t_start
            row = {"epoch": epoch + 1, "env_steps": total_steps, "eval_return": ev["return_mean"],
                   "train_return": last_train_return, "updates": n_updates,
                   "rollout_length": k if cfg.use_model else 0,
                   "time_min": elapsed / 60, "epoch_time_s": time.time() - t_epoch}
            row.update(sac_info)
            row.update(model_stats)
            row.update(rollout_stats)
            if cfg.use_model and len(self.env_buffer) > 0 and total_steps >= cfg.init_random_steps:
                # one-step prediction error on the most recent real transitions
                recent = self.env_buffer.all()
                recent = {k_: v[-cfg.epoch_length:] for k_, v in recent.items()}
                row["model_recent_mse"] = self.model.evaluate_mse(recent)
            self.logger.log(row, total_steps)
            print(f"[{cfg.algo} s{cfg.seed}] epoch {epoch + 1:4d} | steps {total_steps:7d} | "
                  f"eval {ev['return_mean']:9.1f} | train {last_train_return:9.1f} | "
                  f"alpha {sac_info.get('alpha', float('nan')):.3f} | "
                  f"model_mse {model_stats.get('model_holdout_mse', float('nan')):.4f} | "
                  f"k {row['rollout_length']} | {row['epoch_time_s']:.0f}s/epoch | "
                  f"{elapsed / 3600:.2f}h", flush=True)

            if ev["return_mean"] > self.best_eval:
                self.best_eval = ev["return_mean"]
                self.save("best.pt")
            if (epoch + 1) % cfg.save_every_epochs == 0:
                self.save("latest.pt")

            if total_steps >= cfg.total_env_steps:
                break
            if budget is not None:
                # epochs get slower as the dataset grows, so leave room for 1.5 more
                if elapsed + 1.5 * (time.time() - t_epoch) > budget:
                    stop_reason = f"time budget ({cfg.time_budget_hours}h) reached"
                    break

        self.save("final.pt")
        self.logger.close()

        # ---- final, more careful evaluation of the last and the best policy ----
        final_eval = {}
        for name in ("final", "best"):
            path = os.path.join(cfg.out_dir, f"{name}.pt")
            if os.path.exists(path):
                self.agent.load_state_dict(torch.load(path, map_location=self.device)["agent"])
                final_eval[name] = evaluate(self.agent, cfg.env_id, cfg.final_eval_episodes,
                                            seed=777, env=self.eval_env)
        summary = {"algo": cfg.algo, "seed": cfg.seed, "env_steps": total_steps,
                   "best_epoch_eval_return": self.best_eval,
                   "final_policy_return": final_eval["final"]["return_mean"],
                   "final_policy_return_std": final_eval["final"]["return_std"],
                   "best_policy_return": final_eval["best"]["return_mean"],
                   "best_policy_return_std": final_eval["best"]["return_std"],
                   "hours": (time.time() - t_start) / 3600, "stop_reason": stop_reason}
        with open(os.path.join(cfg.out_dir, "summary.json"), "w") as f:
            json.dump(summary, f, indent=2)
        print(f"[{cfg.algo} s{cfg.seed}] done: {summary}", flush=True)
        return summary

**`train.py`** — Command-line entry point used to launch each training run.

In [ ]:
%%writefile train.py
"""Train MBPO (or the SAC baseline) on HalfCheetah-v5.

Examples
--------
    python train.py --algo mbpo --seed 0 --out-dir runs/mbpo_seed0 --time-budget-hours 11
    python train.py --algo sac  --seed 0 --out-dir runs/sac_seed0
    python train.py --algo mbpo --smoke                     # 1-minute sanity check
    python train.py --algo mbpo --set rollout_schedule=[20,100,1,5] --set ensemble_size=5
"""
import argparse
import json

import torch

from mbpo.config import PRESETS, smoke_config
from mbpo.trainer import MBPOTrainer


def parse_args():
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--algo", choices=list(PRESETS), default="mbpo")
    p.add_argument("--seed", type=int, default=0)
    p.add_argument("--device", default="auto")
    p.add_argument("--out-dir", default=None)
    p.add_argument("--total-env-steps", type=int, default=None)
    p.add_argument("--updates-per-step", type=int, default=None, help="SAC updates per env step (G)")
    p.add_argument("--time-budget-hours", type=float, default=None)
    p.add_argument("--smoke", action="store_true", help="tiny config for a quick check")
    p.add_argument("--set", action="append", default=[], metavar="KEY=VALUE",
                   help="override any Config field; VALUE is parsed as JSON when possible")
    return p.parse_args()


def main():
    args = parse_args()
    overrides = {"seed": args.seed, "device": args.device,
                 "out_dir": args.out_dir or f"runs/{args.algo}_seed{args.seed}"}
    for name in ("total_env_steps", "updates_per_step", "time_budget_hours"):
        if getattr(args, name) is not None:
            overrides[name] = getattr(args, name)
    for item in args.set:
        key, value = item.split("=", 1)
        try:
            value = json.loads(value)
        except json.JSONDecodeError:
            pass
        overrides[key] = value

    cfg = smoke_config(args.algo, **overrides) if args.smoke else PRESETS[args.algo](**overrides)
    torch.set_num_threads(1)  # several runs share a few CPU cores on Kaggle
    MBPOTrainer(cfg).train()


if __name__ == "__main__":
    main()

**`record_video.py`** — Renders a policy to mp4.

In [ ]:
%%writefile record_video.py
"""Record an mp4 of a trained policy (or of a random policy for comparison).

MuJoCo needs an off-screen OpenGL backend on a headless machine; set it
*before* running, e.g. `MUJOCO_GL=egl python record_video.py ...`
(fallback: `MUJOCO_GL=osmesa`).

    python record_video.py --checkpoint runs/mbpo_seed0/best.pt --out videos/mbpo.mp4
    python record_video.py --random --out videos/random.mp4
"""
import argparse
import os

import gymnasium as gym
import imageio
import numpy as np
import torch

from mbpo.sac import SAC


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--checkpoint", default=None)
    p.add_argument("--random", action="store_true", help="use a uniformly random policy")
    p.add_argument("--env-id", default="HalfCheetah-v5")
    p.add_argument("--out", default="videos/halfcheetah.mp4")
    p.add_argument("--steps", type=int, default=1000)
    p.add_argument("--seed", type=int, default=0)
    p.add_argument("--width", type=int, default=640)
    p.add_argument("--height", type=int, default=360)
    args = p.parse_args()

    env = gym.make(args.env_id, render_mode="rgb_array", width=args.width, height=args.height)
    obs, _ = env.reset(seed=args.seed)
    env.action_space.seed(args.seed)

    agent = None
    if not args.random:
        ckpt = torch.load(args.checkpoint, map_location="cpu")
        agent = SAC(ckpt["obs_dim"], ckpt["act_dim"], torch.device("cpu"),
                    hidden_dim=ckpt["config"]["hidden_dim"])
        agent.load_state_dict(ckpt["agent"])

    frames, total = [], 0.0
    for _ in range(args.steps):
        frames.append(env.render())
        action = env.action_space.sample() if agent is None else agent.act(obs, deterministic=True)
        obs, r, term, trunc, _ = env.step(action)
        total += r
        if term or trunc:
            break
    env.close()

    os.makedirs(os.path.dirname(os.path.abspath(args.out)), exist_ok=True)
    fps = env.metadata.get("render_fps", 20)
    imageio.mimsave(args.out, [np.asarray(f) for f in frames], fps=fps, macro_block_size=1)
    print(f"saved {args.out}: {len(frames)} frames, return {total:.1f}")


if __name__ == "__main__":
    main()

**`plot_results.py`** — Learning curves and the results table.

In [ ]:
%%writefile plot_results.py
"""Plot learning curves from runs/*/progress.csv.

    python plot_results.py --runs runs --out results

Produces
  learning_curves.png  eval return vs env steps (mean over seeds, min-max band)
  model_loss.png       dynamics-model holdout MSE vs env steps (MBPO runs)
  results_table.csv    final numbers per run (from summary.json)
"""
import argparse
import glob
import json
import os

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

COLORS = {"mbpo": "#2a78d6", "sac": "#eb6834"}   # fixed colour per algorithm
LABELS = {"mbpo": "MBPO (ours)", "sac": "SAC (model-free baseline)"}
INK, INK2, GRID, SURFACE = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb"


def load_runs(root):
    runs = []
    for csv_path in sorted(glob.glob(os.path.join(root, "*", "progress.csv"))):
        run_dir = os.path.dirname(csv_path)
        with open(os.path.join(run_dir, "config.json")) as f:
            cfg = json.load(f)
        df = pd.read_csv(csv_path)
        if len(df):
            runs.append({"dir": run_dir, "algo": cfg["algo"], "seed": cfg["seed"], "df": df})
    return runs


def style(ax, title, ylabel):
    ax.set_facecolor(SURFACE)
    ax.set_title(title, loc="left", color=INK, fontsize=13, fontweight="bold")
    ax.set_xlabel("environment steps", color=INK2)
    ax.set_ylabel(ylabel, color=INK2)
    ax.grid(True, color=GRID, linewidth=0.8)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color(GRID)
    ax.tick_params(colors=INK2)
    ax.xaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(
        lambda x, _: f"{x / 1000:.0f}k" if x else "0"))


def smooth(y, w):
    return pd.Series(y).rolling(w, min_periods=1).mean().to_numpy()


def curve(ax, runs, algo, col, w):
    group = [r for r in runs if r["algo"] == algo and col in r["df"]]
    if not group:
        return
    n = min(len(r["df"]) for r in group)  # align seeds on the shortest run
    x = group[0]["df"]["env_steps"].to_numpy()[:n]
    ys = np.stack([smooth(r["df"][col].to_numpy()[:n], w) for r in group])
    c = COLORS.get(algo, "#6250d6")
    label = f"{LABELS.get(algo, algo)} ({len(group)} seed{'s' if len(group) > 1 else ''})"
    if len(group) > 1:
        ax.fill_between(x, ys.min(0), ys.max(0), color=c, alpha=0.18, linewidth=0)
    ax.plot(x, ys.mean(0), color=c, linewidth=2, label=label)
    ax.annotate(f"{ys.mean(0)[-1]:.0f}", (x[-1], ys.mean(0)[-1]), xytext=(6, 0),
                textcoords="offset points", color=INK, fontsize=10, va="center")


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--runs", default="runs")
    p.add_argument("--out", default="results")
    p.add_argument("--smooth", type=int, default=5, help="moving-average window (epochs)")
    args = p.parse_args()
    os.makedirs(args.out, exist_ok=True)
    runs = load_runs(args.runs)
    if not runs:
        print("no runs found")
        return
    algos = [a for a in ("mbpo", "sac") if any(r["algo"] == a for r in runs)]

    fig, ax = plt.subplots(figsize=(9, 5), facecolor=SURFACE)
    for algo in algos:
        curve(ax, runs, algo, "eval_return", args.smooth)
    style(ax, "HalfCheetah-v5: evaluation return", "return (deterministic policy)")
    ax.legend(frameon=False, labelcolor=INK)
    fig.tight_layout()
    fig.savefig(os.path.join(args.out, "learning_curves.png"), dpi=150)
    plt.close(fig)

    if "mbpo" in algos:
        fig, ax = plt.subplots(figsize=(9, 4), facecolor=SURFACE)
        curve(ax, runs, "mbpo", "model_holdout_mse", 1)
        style(ax, "Dynamics model: holdout MSE (elites)", "MSE on held-out real data")
        ax.set_yscale("log")
        ax.legend(frameon=False, labelcolor=INK)
        fig.tight_layout()
        fig.savefig(os.path.join(args.out, "model_loss.png"), dpi=150)
        plt.close(fig)

    rows = []
    for r in runs:
        s_path = os.path.join(r["dir"], "summary.json")
        row = {"run": os.path.basename(r["dir"]), "algo": r["algo"], "seed": r["seed"],
               "env_steps": int(r["df"]["env_steps"].iloc[-1]),
               "last_eval_return": float(r["df"]["eval_return"].iloc[-1]),
               "max_eval_return": float(r["df"]["eval_return"].max())}
        if os.path.exists(s_path):
            with open(s_path) as f:
                s = json.load(f)
            row.update({k: s[k] for k in ("final_policy_return", "best_policy_return", "hours") if k in s})
        rows.append(row)
    table = pd.DataFrame(rows)
    table.to_csv(os.path.join(args.out, "results_table.csv"), index=False)
    print(table.to_string(index=False))

    # sample efficiency: env steps needed to first reach a return threshold
    for thr in (3000, 5000, 8000):
        hits = {}
        for r in runs:
            ok = r["df"][r["df"]["eval_return"] >= thr]
            hits.setdefault(r["algo"], []).append(int(ok["env_steps"].iloc[0]) if len(ok) else None)
        print(f"steps to reach return {thr}: {hits}")


if __name__ == "__main__":
    main()

## Rendering check and a random-policy video (the "before" clip)

In [ ]:
# MuJoCo needs an off-screen OpenGL backend on a headless server: try EGL (GPU), then OSMesa (CPU).
import subprocess, sys, os

def try_backend(backend):
    test = ("import gymnasium as gym; e = gym.make('HalfCheetah-v5', render_mode='rgb_array'); "
            "e.reset(seed=0); print(e.render().shape)")
    r = subprocess.run([sys.executable, "-c", test], capture_output=True, text=True,
                       env={**os.environ, "MUJOCO_GL": backend}, timeout=300)
    return r.returncode == 0, (r.stdout + r.stderr).strip().splitlines()[-1:]

RENDER_BACKEND = None
for backend in ["egl", "osmesa", "glfw"]:  # glfw only works with a display (local runs)
    ok, msg = try_backend(backend)
    print(f"MUJOCO_GL={backend}: {'OK' if ok else 'failed'} {msg}")
    if ok:
        RENDER_BACKEND = backend
        break
if RENDER_BACKEND is None:  # install OSMesa and retry
    subprocess.run("apt-get -qq update && apt-get -qq install -y libosmesa6 libosmesa6-dev > /dev/null",
                   shell=True)
    ok, msg = try_backend("osmesa")
    print("after installing libosmesa6:", "OK" if ok else "failed", msg)
    RENDER_BACKEND = "osmesa" if ok else None
print("render backend:", RENDER_BACKEND, "(training does not need rendering; only the videos do)")

def record(args):
    if RENDER_BACKEND is None:
        print("skipping video: no rendering backend")
        return
    r = subprocess.run([sys.executable, "record_video.py", *args], capture_output=True, text=True,
                       env={**os.environ, "MUJOCO_GL": RENDER_BACKEND})
    if r.returncode == 0:
        print(r.stdout.strip().splitlines()[-1])
    else:
        print(f"video failed (exit {r.returncode}):", r.stderr[-1500:])

record(["--random", "--out", "videos/random_policy.mp4"])   # "before training" clip

## Smoke test

In [ ]:
# Fast end-to-end check of the whole pipeline before spending hours on the real run.
import subprocess, sys
r = subprocess.run([sys.executable, "train.py", "--algo", "mbpo", "--smoke", "--out-dir", "smoke/mbpo"],
                   capture_output=True, text=True)
print("\n".join(l for l in r.stdout.splitlines() if l.startswith("[")))
if r.returncode != 0:
    print(r.stderr[-3000:])
    raise RuntimeError("smoke test failed - fix this before launching the long run")
print("smoke test passed")

## Training
Every run is a separate process (`train.py`), pinned to one GPU with `CUDA_VISIBLE_DEVICES`.
Progress is logged once per epoch (= 1000 env steps) to `runs/<run>/progress.csv`, TensorBoard
(`runs/<run>/tb`) and `runs/<run>/stdout.log`; this cell prints a status line every 15 minutes.
Each run stops by itself when `TRAIN_HOURS` is used up, saves `best.pt` / `final.pt`, and evaluates
both policies on 10 episodes (`summary.json`).

In [ ]:
import subprocess, sys, time, os, json

def launch(algo, seed, gpu):
    out = f"runs/{algo}_seed{seed}"
    os.makedirs(out, exist_ok=True)
    cmd = [sys.executable, "train.py", "--algo", algo, "--seed", str(seed), "--out-dir", out,
           "--time-budget-hours", str(TRAIN_HOURS)]
    if SMOKE_TEST:
        cmd += ["--smoke"]
    else:
        cmd += ["--total-env-steps", str(TOTAL_ENV_STEPS)]
        if algo == "mbpo":
            cmd += ["--updates-per-step", str(UPDATES_PER_STEP)]
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu) if N_GPU else ""}
    log = open(f"{out}/stdout.log", "w")
    print(f"launch {out} on GPU {gpu if N_GPU else 'none (CPU)'}")
    return out, subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, env=env)

plan = [("mbpo", s, i % max(N_GPU, 1)) for i, s in enumerate(MBPO_SEEDS)]
if RUN_SAC_BASELINE:
    plan.append(("sac", 0, 0))
procs = [launch(*p) for p in plan]

def last_epoch_line(out):
    try:
        lines = [l for l in open(f"{out}/stdout.log") if "epoch" in l]
        return lines[-1].strip() if lines else "(starting...)"
    except FileNotFoundError:
        return "(no log yet)"

t0, last_print = time.time(), -1e9
while any(p.poll() is None for _, p in procs):
    if time.time() - last_print > (60 if SMOKE_TEST else 900):
        last_print = time.time()
        print(f"--- {(time.time() - t0) / 3600:.2f} h elapsed ---")
        for out, p in procs:
            print(("running " if p.poll() is None else "finished") + " | " + last_epoch_line(out))
    time.sleep(10)

print(f"all runs finished after {(time.time() - t0) / 3600:.2f} h")
for out, p in procs:
    status = "OK" if p.returncode == 0 else f"FAILED (exit {p.returncode})"
    print(f"{out}: {status}")
    if p.returncode != 0:
        print("".join(open(f"{out}/stdout.log").readlines()[-30:]))
    elif os.path.exists(f"{out}/summary.json"):
        print("   ", json.load(open(f"{out}/summary.json")))

## Results
Learning curves (evaluation return of the deterministic policy after every 1000 env steps, smoothed over
5 epochs; shaded band = min-max over seeds), the dynamics-model error, and the final table.

In [ ]:
import subprocess, sys
from IPython.display import Image, display
print(subprocess.run([sys.executable, "plot_results.py", "--runs", "runs", "--out", "results"],
                     capture_output=True, text=True).stdout)
for name in ["learning_curves.png", "model_loss.png"]:
    if os.path.exists(f"results/{name}"):
        display(Image(f"results/{name}"))

## Videos

In [ ]:
# Video of the best MBPO policy (chosen by its 10-episode evaluation in summary.json).
import glob, json, base64
from IPython.display import HTML, display

best = None
for s_path in glob.glob("runs/mbpo_seed*/summary.json"):
    s = json.load(open(s_path))
    for which in ("best", "final"):
        score = s[f"{which}_policy_return"]
        if best is None or score > best[0]:
            best = (score, os.path.join(os.path.dirname(s_path), f"{which}.pt"))
if best:
    print(f"recording {best[1]} (10-episode mean return {best[0]:.0f})")
    record(["--checkpoint", best[1], "--out", "videos/mbpo_trained.mp4"])
sac_ckpt = "runs/sac_seed0/best.pt"
if os.path.exists(sac_ckpt):
    record(["--checkpoint", sac_ckpt, "--out", "videos/sac_baseline.mp4"])

for v in ["videos/random_policy.mp4", "videos/mbpo_trained.mp4"]:
    if os.path.exists(v):
        b64 = base64.b64encode(open(v, "rb").read()).decode()
        display(HTML(f"<p><b>{v}</b></p><video width=640 controls src='data:video/mp4;base64,{b64}'></video>"))

## Download

In [ ]:
# Bundle everything for download (Output tab -> mbpo_results.zip).
import shutil, tempfile
staging = tempfile.mkdtemp()
for d in ["runs", "results", "videos"]:
    if os.path.isdir(d):
        shutil.copytree(d, os.path.join(staging, d))
shutil.make_archive("mbpo_results", "zip", staging)
print(f"mbpo_results.zip: {os.path.getsize('mbpo_results.zip') / 1e6:.1f} MB")